# Mathematical Solver

Solver

TODO:
- Modelling
    - Your current method is: Path-based + Edge-local
    - Wiedemann is: Flow-based + Global rerouting
- Seperate breakdown for main roads and side streets?
- All the other relexxations? (such as fixing roads)
- Planners could incorporate Sheng Liu’s General Utility (GU) function—which rewards the size of continuous segments—directly into Wiedmann’s multi-criteria objective functio

- Do some seperate logic for main roads to allow for whole dedicated bike lanes instead of fietsstraat?


- allow for direction changing of cars


- Unlike the Barahimi or Mathew models, which use BPR functions where travel time increases as capacity decreases (t=f(capacity)), Wiedmann uses fixed travel times (tc​,tb​) based on distance and speed limits

- consider car flow for segment selection?

- hypervolume indicator (HI) to compare final results.

Known Limitations:
- Number of lanes is not considered for travel time, and reallocation cost, since congestion is not being simulated explicitly. Techinically I could create some 'friction' factor were if there a multiple lanes, than conversion to fietsstraat,creates more 'friction'



idk man: 
- so wiedmann seperated the main road and side street solve, so that he could parrallelise across different cities. Since main roads cross across many cities. 

- I could technincally do different logic for this main road where i can build dedicated bike segments instead of fietstraatt.

- Note i considered an approach where we would have flow for dedicated car ,dedicated bike, fieetstrrat and shared bike guests, but this would make the decisions too complex for the solver

Conclusions:
- if im going to do the parrallelisation (i prob am), ig i should do the seperate dedicated logic but this complicates stuff? but tbh the realism is needed.

- if im doing it all in one pass, i can just feed the solver my special logic, and it wouldn't know its special it just picks from the options i give it.

conclusion's conclusion:
- I want to go for all in one pass, but i dont think its computaitonally possible for malta

Methodology:

**City-wide lane reallocation process in Wiedemann et al.**

The city-scale lane reallocation framework proposed by Wiedemann et al. follows a structured, multi-stage optimisation workflow designed to remain tractable for large urban networks.

1. Network extraction and preprocessing
The street network is extracted from OpenStreetMap (OSM) and represented as a graph with edges annotated by length, gradient, speed limit, and lane capacity. Infrastructure that cannot be modified, such as highways or tram corridors, is fixed and excluded from reallocation.

1. Lane-level graph construction
The base undirected street graph is converted into a directed multigraph to enable direction-specific and lane-level capacity allocation. This representation allows lanes to be reassigned between car and bicycle use independently in each direction.

1. Definition of travel-time impedance
Car travel time is computed from distance and speed limits, while bicycle travel time incorporates additional penalties reflecting perceived safety, gradient, and riding on non-protected (car) infrastructure.

1. Demand representation
Rather than an all-pairs formulation, the model uses a demand-driven origin–destination (OD) set to limit computational complexity while capturing dominant travel flows across the city.

1. Separation of main and non-main roads
The network is divided into main roads and non-main roads. This separation reduces problem size and ensures that the primary car network is treated more conservatively during lane reallocation.

1. Partitioning into independent regions
Main roads are partitioned into five large regions, while non-main roads are partitioned into 57 smaller regions, resulting in 62 regional optimisation problems.

1. Iterative LP-based lane allocation (per region)

    For each region, an iterative optimisation procedure is applied:

    - solve a linear program with relaxed lane-allocation variables,

    - fix the largest or most decisive lane allocations,

    - re-solve the LP with a reduced decision space,

    - repeat until convergence or until any further bicycle-lane allocation would disconnect the regional car network.

1. Parallel execution
Each regional optimisation problem is solved independently. These instances are executed in parallel using Python’s multiprocessing framework, substantially reducing total runtime.

1. City-wide solution assembly
The regional lane-allocation results are merged to form a consistent city-wide lane reallocation plan, specifying which lanes are dedicated to bicycles and which remain available for cars in each direction.

1. Evaluation of final network performance
The resulting network is evaluated using bicycle and car travel-time metrics under the defined OD demand, and alternative policy scenarios are explored by varying the objective-function weights.


**Methodology Mine**

1. Repeat until budget is spent:
1. Compute bike shortest paths on current bike weights
1. Compute car shortest paths on current car weights
1. Solve a locality-constrained edge selection problem for a small batch k_batch
1. Apply the selected reallocations to the master graph
1. Recompute costs (bike and car) for affected edges
1. Loop

In [ ]:
# By solving in multiple batches, we reduce computation time
#TODO CHatGPT is not agreeing, saying that recalculting the OD paths is way more expensive and takes the most time. And that my formulation is actually quite simple????


In [ ]:
# Build LP inputs for lane-level rounding,
eligible_bike_arcs = set(graph_util.make_reallocatable_subgraph(G_master_unsimplified).edges(keys=True))
main_arcs = set(graph_util.make_highway_subgraph(G_master_unsimplified, "primary").edges(keys=True))
lambda_seg = {
    seg: max(1.0, sum(float(G_master_unsimplified[u][v][k].get("car_lanes", 1.0)) for (u, v, k) in arcs))
    for seg, arcs in seg_to_arcs.items()
}

car_arcs = set(graph_util.make_drive_subgraph(G_master_unsimplified).edges(keys=True))
bike_arcs = set(graph_util.make_bikeable_subgraph(G_master_unsimplified).edges(keys=True))

od_weighted = ODGeneration.scale_OD_pairs(od, bike_share=0.1)
od_weighted = ODGeneration.append_auxiliary_chain_od_pairs(od_weighted, rng, list(G_drive.nodes()), shuffle_nodes = False)

G_bike = graph_util.make_bikeable_subgraph(G_master_unsimplified)
od_allowed_arcs_car, od_allowed_arcs_bike = paths_util.build_od_allowed_arcs(
    G_drive=G_drive,
    G_bike=G_bike,
    OD_list=od_weighted,
    corridor_hops=30,
)



TODO
Second mismatch: you are forcing the same OD demand for cars and bikes
- auxilery paths?
- check if my OD pair change broke somehting somwhere

---------
So:

ω-weighting = “importance weighting” (priority in objective)

RHS scaling = “volume weighting” (priority and capacity consumption)

Wiedemann defaults to the first; they acknowledge the second as an extension.

In [ ]:
def build_region_lp_inputs(
    *,
    G_master,
    region_name: str,
    OD_full:OD,
    region_key: Literal["localities", "regions", "highway"] = "localities",
    od_filter_mode:Literal["internal", "touching"] = "internal",
    fixed_bike_1_global=None,
    fixed_bike_0_global=None,
):
    """
    Returns:
    eligible_bike_arcs, fixed_bike_1, fixed_bike_0, OD_region
    """
    fixed_bike_1_global = set(fixed_bike_1_global or set())
    fixed_bike_0_global = set(fixed_bike_0_global or set())

    # 1) region scope only for decision filtering
    if region_key == "localities":
        G_region = graph_util.make_locality_subgraph(G_master, region_name)
    elif region_key == "regions":
        G_region = graph_util.make_region_subgraph(G_master, region_name)
    elif region_key == "highway":
        G_region = graph_util.make_highway_subgraph(G_master, region_name)
    else:
        raise ValueError("region_key must be one of: localities, regions, highway")

    region_reallocatable_arcs = set(
        graph_util.make_reallocatable_subgraph(G_region).edges(keys=True)
    )
    print(f"There are {len(region_reallocatable_arcs)} reallocatable arcs in this region")
    global_reallocatable_arcs = set(
        graph_util.make_reallocatable_subgraph(G_master).edges(keys=True)
    )

    # 2) global fixed sets (cleaned to existing arcs)
    global_arc_set = set(G_master.edges(keys=True))
    fixed_bike_1 = {a for a in fixed_bike_1_global if a in global_arc_set}
    fixed_bike_0 = {a for a in fixed_bike_0_global if a in global_arc_set}

    # 3) freeze all reallocatable arcs outside region
    outside_region_reallocatable = global_reallocatable_arcs - region_reallocatable_arcs
    fixed_bike_0 |= outside_region_reallocatable

    # avoid contradictions
    fixed_bike_0 -= fixed_bike_1

    # 4) eligible = region decision arcs + already-fixed-to-1 arcs
    eligible_bike_arcs = region_reallocatable_arcs | fixed_bike_1

    # OD filtering policy
    region_nodes = set(G_region.nodes())
    if od_filter_mode == "all":
        OD_region = list(OD_full)
    elif od_filter_mode == "internal":
        OD_region = [
            od for od in OD_full
            if od.origin in region_nodes and od.destination in region_nodes
        ]
    elif od_filter_mode == "touching":
        OD_region = [
            od for od in OD_full
            if od.origin in region_nodes or od.destination in region_nodes
        ]
    else:
        raise ValueError("od_filter_mode must be 'all', 'internal', or 'touching'")

    # sanity checks
    #assert fixed_bike_1.isdisjoint(fixed_bike_0)
    #assert fixed_bike_1.issubset(eligible_bike_arcs)

    return eligible_bike_arcs, fixed_bike_1, fixed_bike_0, OD_region

In [ ]:
eligible_bike_arcs, fixed_bike_1_region, fixed_bike_0_region, OD_region = build_region_lp_inputs(
    G_master=G_master_unsimplified,
    region_name="primary",
    region_key="highway",
    OD_full=od_weighted,
    fixed_bike_1_global= None,
    fixed_bike_0_global=None,
    od_filter_mode="internal",
)

In [ ]:
"""bad = []
for p, od in enumerate(od_weighted):
    arcs = od_allowed.get(p, set())
    if len(arcs) == 0:
        bad.append((p, od.origin, od.destination, od.is_auxiliary, od.bike_weight, od.car_weight))
bad[:20], len(bad)"""

In [ ]:
"""bad = []
for key, allowed_arcs in od_allowed.items():
    if len(allowed_arcs) == 0:
        bad.append((allowed_arcs))
bad[:20], len(bad)"""

In [ ]:
"""P = len(od_weighted)
keys = set(od_allowed.keys())
missing = sorted(set(range(P)) - keys)
extra = sorted(keys - set(range(P)))
print("P:", P, "keys:", len(keys), "missing:", len(missing), "extra:", len(extra))
print("first missing:", missing[:10], "first extra:", extra[:10])
"""

In [ ]:
import networkx as nx

def audit_od(OD_lp, od_allowed_lp):
    bad = []
    for p, od in enumerate(OD_lp):
        arcs = set(od_allowed_lp.get(p, set()))
        s, t = od.origin, od.destination
        out_s = sum(1 for u,v,k in arcs if u == s)
        in_t  = sum(1 for u,v,k in arcs if v == t)
        H = nx.DiGraph((u,v) for u,v,k in arcs)
        has_path = (s in H and t in H and nx.has_path(H, s, t))
        if (not arcs) or (out_s == 0) or (in_t == 0) or (not has_path):
            bad.append((p, s, t, od.is_auxiliary, len(arcs), out_s, in_t, has_path))
    return bad

#bad = audit_od(od_weighted, od_allowed)
#print("bad ODs:", len(bad))
#bad[:20]


In [ ]:
"""p = 67  # replace with IIS p
od = OD_region[p]
arcs = od_allowed.get(p, set())
print("OD:", p, od.origin, od.destination, od.is_auxiliary, "arcs:", len(arcs))
print("out_s:", sum(1 for u,v,k in arcs if u == od.origin))
print("in_t :", sum(1 for u,v,k in arcs if v == od.destination))
"""

In [ ]:
for nid in [158594371, 11987707711]:
    print(nid, G_master_unsimplified.nodes[nid])

In [ ]:
edges_to_check = [
    (158594371, 1497329695, 0),
    (1497329695, 158594371, 0),
    (158594371, 150608097, 0),
    (150608097, 158594371, 0),
    (158594371, 306268490, 0),
    (306268490, 158594371, 0),
    (158594371, 306268492, 0),
    (306268492, 158594371, 0),
]

for u, v, k in edges_to_check:
    print(u, v, k, G_master_unsimplified.edges[u, v, k])

In [ ]:
sub = nx.ego_graph(G_master_unsimplified, 158594371, radius=1)
ox.plot_graph(sub, node_color='orange', node_size=50)

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt

# your two nodes
source = 158594371
target = 11987707711

# plot the base graph
fig, ax = ox.plot_graph(G_master_unsimplified, show=False, close=False, figsize=(40,40))

# get coordinates of the special nodes
sx = G_master_unsimplified.nodes[source]['x']
sy = G_master_unsimplified.nodes[source]['y']
tx = G_master_unsimplified.nodes[target]['x']
ty = G_master_unsimplified.nodes[target]['y']

# overlay highlighted nodes
ax.scatter([sx], [sy], c='red', s=50, zorder=5, label='Source')
ax.scatter([tx], [ty], c='red', s=50, marker='s', zorder=5, label='Target')

# optional: add legend
ax.legend()

plt.show()


In [ ]:
#TODO had to keep other non region edges, due to needing flow sinks? but maybe this can be minimised by taking main street only (prob wont work), buffer area. 

#TODO create a buffer region, that is just only add flow variables for edges that are in OD_allowed union for all trips in that region ig. they are not deleted from the graph, but they are removed from the LP routing polytope (for that OD) by not instantiating flow vars.

#TODO am i passing the G_bicycle unique edges as fixed bike lanes. as well as protected bike lanes.

fixed_main_1, fixed_main_0, hist_main = mathematical.round_lp_solution_segment_aware(
    G=G_master_unsimplified,
    OD=OD_region,
    seg_to_arcs=seg_to_arcs,
    arc_to_seg=arc_to_seg,
    Lambda_seg=lambda_seg,
    eligible_bike_arcs=eligible_bike_arcs,
    gamma=2.0,
    k_fix=25,
    budget_bike_lanes=10_000,
    od_allowed_arcs_bike=od_allowed_arcs_bike,
    od_allowed_arcs_car=od_allowed_arcs_car,
    car_arcs=car_arcs,
    bike_arcs=bike_arcs,
    bike_shared_cost_attr="bike_cost_penalty",
    bike_dedicated_cost_attr="bike_cost_base",
    fixed_bike_0_init= fixed_bike_0_region ,
    fixed_bike_1_init= fixed_bike_1_region,
    max_relative_deterioration = 0.2,
    verbose=True,
    print_problem_stats=True,
)

display(hist_main)
display(fixed_main_1)
display(fixed_main_0)


Current problem:

I want to do a baseline objective score, so that i can measure the relative error of my changes. Now recall that i am doing a main road pass and then multiple seperate region solves, and that in the region subproblem the solver only see's the objective score of that subproblem. Therefore I would have to get an baseline objective score of each sub region after the main fixing. But then how would I account for the relative damage i already caused? 

In [ ]:
from Plotting.renderer import draw_graph
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_fixed_arcs(G, fixed_arcs_bike:set, fixed_arcs_car:set, title="Fixed arcs from rounding"):
    fixed_arcs_bike = set(fixed_arcs_bike)

    edge_colors = []
    edge_widths = []
    for u, v, k in G.edges(keys=True):
        arc = (u, v, k)
        if arc in fixed_arcs_bike:
            edge_colors.append("limegreen")
            edge_widths.append(1.8)
        elif arc in fixed_arcs_car:
            edge_colors.append("red")
            edge_widths.append(1)
        else:
            edge_colors.append("lightgray")
            edge_widths.append(0.35)

    fig, ax = draw_graph(
        G,
        edge_color=edge_colors,
        edge_linewidth=edge_widths,
        node_size=0,
        fig_size=(12, 12),
        bgcolor="white",
    )

    ax.legend(
        handles=[
            mpatches.Patch(color="limegreen", label=f"Fixed bike arcs ({len(fixed_arcs_bike)})"),
            mpatches.Patch(color="red", label=f"Fixed car arcs ({len(fixed_arcs_car)})"),
            mpatches.Patch(color="lightgray", label="Other edges"),
        ],
        loc="best",
    )
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()
    plt.show()
    return fig, ax

# after your rounding call
plot_fixed_arcs(G_drive, fixed_main_1, fixed_main_0, title="Round LP fixed edges (primary)")


In [ ]:
import re

def _iis_fix_b1_arcs(iis_path="flow_lane_lp_iis.ilp"):
    print("FRAUDING")
    bad = set()
    pattern = re.compile(r"fix_b1_(-?\d+)_(-?\d+)_(-?\d+)")
    try:
        with open(iis_path, "r", encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                m = pattern.search(line)
                if m:
                    bad.add((int(m.group(1)), int(m.group(2)), int(m.group(3))))
    except OSError:
        pass
    return bad

fixed_global_1 = set(fixed_main_1)
fixed_global_0 = set(fixed_main_0) | (main_arcs - fixed_main_1)  # lock all unfixed primary arcs

# Pass 2: locality solves (sequential, keeping global fixings consistent across regions)
region_budget_increment = 200
max_locality_retries = 3
history_by_region = {}

for region_name in subgraphs:
    retries = 0
    while True:
        eligible_region_arcs, fixed_seed_1, fixed_seed_0, OD_region = build_region_lp_inputs(
            G_master=G_master_unsimplified,
            region_name=region_name,
            region_key="localities",
            OD_full=od_weighted,
            od_filter_mode="internal",
            fixed_bike_1_global=fixed_global_1,
            fixed_bike_0_global=fixed_global_0,
        )

        if not OD_region or not eligible_region_arcs:
            history_by_region[region_name] = []
            break

        try:
            new_fixed_1, new_fixed_0, hist_region = mathematical.round_lp_solution_segment_aware(
                G=G_master_unsimplified,
                OD=OD_region,
                seg_to_arcs=seg_to_arcs,
                arc_to_seg=arc_to_seg,
                Lambda_seg=lambda_seg,
                eligible_bike_arcs=eligible_region_arcs,
                car_arcs=car_arcs,
                bike_arcs=bike_arcs,
                gamma=2.0,
                k_fix=25,
                budget_bike_lanes=len(fixed_seed_1) + region_budget_increment,
                fixed_bike_1_init=fixed_seed_1,
                fixed_bike_0_init=fixed_seed_0,
                od_allowed_arcs_bike=od_allowed_arcs_bike,
                od_allowed_arcs_car=od_allowed_arcs_car,
                bike_shared_cost_attr="bike_cost_penalty",
                bike_dedicated_cost_attr="bike_cost_base",
                max_relative_deterioration=0.2,
                verbose=False,
                print_problem_stats=False,
            )
        except RuntimeError as exc:
            if "infeasible" not in str(exc).lower() or retries >= max_locality_retries:
                raise
            bad_arcs = _iis_fix_b1_arcs("flow_lane_lp_iis.ilp") & set(eligible_region_arcs)
            if not bad_arcs:
                raise
            fixed_global_1 -= bad_arcs
            fixed_global_0 |= bad_arcs
            fixed_global_0 -= fixed_global_1
            retries += 1
            print(f"[{region_name}] infeasible; blocked {len(bad_arcs)} IIS fix_b1 arcs and retrying ({retries}/{max_locality_retries})")
            continue

        fixed_global_1 |= new_fixed_1
        fixed_global_0 |= new_fixed_0
        fixed_global_0 -= fixed_global_1
        history_by_region[region_name] = hist_region
        break

fixed_city_1 = fixed_global_1
fixed_city_0 = fixed_global_0

print(f"Pass 2 complete across {len(subgraphs)} localities")
print(f"fixed_city_1 (bike): {len(fixed_city_1)}")
print(f"fixed_city_0 (car): {len(fixed_city_0)}")


In [ ]:
plot_fixed_arcs(G_drive, fixed_city_1, fixed_city_0, title="Round LP fixed edges (primary)")
